# SmartHome claim index: what the corpus claims, and how well it is evidenced

**Claim cluster:**   
**Source of truth:** `data/smarthome.duckdb` (read-only), via `src/viz_core.py`

In [ ]:
import os
import sys
from pathlib import Path

import ipywidgets as widgets
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from IPython.display import HTML, clear_output, display

# Find the repo root by walking up until schemas/stage2.sql appears, so the
# notebook runs from notebooks/, from the repo root, or from anywhere else.
_here = Path.cwd().resolve()
for _candidate in [_here, *_here.parents]:
    if (_candidate / "schemas" / "stage2.sql").exists():
        ROOT = _candidate
        break
else:  # pragma: no cover - only reachable outside a checkout
    raise FileNotFoundError("schemas/stage2.sql not found above " + str(_here))

if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

# Option A topology: this wave reads one dedicated Stage 2 database. Point
# viz_core at it before import -- it resolves the path once at import time --
# so every fetch in this kernel uses the same file.
os.environ["SMARTHOME_DB"] = str(ROOT / "data" / "smarthome.duckdb")

import viz_core as viz  # noqa: E402

pd.set_option("display.max_colwidth", 90)
pd.set_option("display.width", 160)

# Renderers matter for two reasons. "browser" (plotly's default) opens a
# window when a figure is shown outside a live kernel, which breaks headless
# execution. The bare "notebook" renderer in plotly 7 emits only text/html,
# carrying no figure payload, so a notebook could run clean and still show
# nothing. "plotly_mimetype+notebook" emits the figure JSON for JupyterLab and
# an inline HTML fallback elsewhere -- which is also what the headless test
# asserts on.
pio.renderers.default = "plotly_mimetype+notebook"

# Every figure in this notebook reads only from data/smarthome.duckdb (read-only).
CLAIM_IDS = ()
assert viz.DEFAULT_DB_PATH.endswith("smarthome.duckdb"), viz.DEFAULT_DB_PATH

claims = viz.fetch_claims(list(CLAIM_IDS))
sources = viz.sources_for_claims(list(CLAIM_IDS)) if CLAIM_IDS else pd.DataFrame()
cluster_metrics = (
    viz.claim_metrics_lookup(list(CLAIM_IDS)) if CLAIM_IDS else pd.DataFrame()
)

# Corpus-wide row counts, read once so a notebook can state the size of what it
# is drawing from without confusing a whole-corpus count with the size of its
# own (possibly empty) claim cluster.
CORPUS = viz.connect_db()
try:
    corpus_counts = {
        table: CORPUS.execute(f"SELECT COUNT(1) FROM {table}").fetchone()[0]
        for table in ("stage2_claim", "metric", "entity", "source_mirror",
                      "predicate", "extraction_decision")
    }
finally:
    CORPUS.close()

def provenance(claim_id):
    """Collapsible evidence panel for one claim in this cluster."""
    panel = viz.render_provenance_panel(claim_id)
    return widgets.VBox([panel])

def show(*items):
    """Display data frames, HTML, or figures, one per argument."""
    for item in items:
        display(item)

def confidence_legend():
    """Show the colour coding applied to confidence across all figures."""
    display(HTML(
        "<b>Confidence colour coding:</b> "
        f"<span style='color:{viz.CONFIDENCE_COLORS['high']}'>high</span> &middot; "
        f"<span style='color:{viz.CONFIDENCE_COLORS['medium']}'>medium</span> &middot; "
        f"<span style='color:{viz.CONFIDENCE_COLORS['low']}'>low</span>"
    ))

## Provenance

Every figure on this page is computed from the full claim table, not from a curated subset. The evidence-class distribution below is the honest summary of the corpus: most of it is reported signal, not documented fact.

In [ ]:
for claim_id in CLAIM_IDS:
    display(HTML(f"<h4>{claim_id} &mdash; {claims.set_index('local_id').loc[claim_id, 'claim_text']}</h4>"))
    display(provenance(claim_id))
confidence_legend()

## Primary evidence

### Evidence class and confidence across all 119 claims

Colour is confidence; the y-axis is the claim's evidence class as recorded at extraction. The point of the chart is the shape: a large `reported signal` band at medium confidence means much of the corpus is vendor or analyst material relayed as a claim, which is why the extraction log contains `evidence_downgrade` decisions.

In [ ]:
all_claims = viz.fetch_all_claims()
display(HTML(
    f"<p>Corpus totals: <b>{len(all_claims)}</b> claims, "
    f"<b>{corpus_counts['metric']}</b> metrics, "
    f"<b>{corpus_counts['entity']}</b> entities, "
    f"<b>{corpus_counts['source_mirror']}</b> sources, "
    f"<b>{corpus_counts['predicate']}</b> predicates, "
    f"<b>{corpus_counts['extraction_decision']}</b> extraction decisions. "
    "Every figure on this page is a group-by over the full claim table.</p>"
))
show(all_claims[["local_id", "claim_type", "evidence_marker", "confidence",
                 "workflow_stage", "source_refs", "section"]].head(20))

_counts = (all_claims.groupby(["claim_type", "confidence"]).size()
           .reset_index(name="claims"))
_types = list(dict.fromkeys(_counts["claim_type"]))
_confs = ["high", "medium", "low"]

fig = go.Figure()
for _conf in _confs:
    _slice = _counts[_counts["confidence"] == _conf].set_index("claim_type")
    fig.add_trace(go.Bar(
        name=_conf,
        y=[_slice.loc[t, "claims"] if t in _slice.index else 0 for t in _types],
        x=_types,
        marker_color=viz.CONFIDENCE_COLORS[_conf],
        customdata=[[t, _conf] for t in _types],
        hovertemplate="%{customdata[0]}<br>confidence: %{customdata[1]}"
                      "<br>%{y} claims<extra></extra>",
    ))
fig.update_layout(
    title="Claims by evidence class and confidence",
    barmode="stack", yaxis_title="claims", xaxis_title="evidence class",
    height=380,
)
fig.show()

domain_counts = (all_claims.groupby("domain").size()
                 .sort_values(ascending=False).reset_index(name="claims"))
display(HTML("<h4>Claims per notebook domain</h4>"))
show(domain_counts)
fig = go.Figure(go.Bar(
    x=domain_counts["claims"], y=domain_counts["domain"], orientation="h",
    marker_color="#4a6fa5",
    customdata=domain_counts["domain"],
    hovertemplate="%{customdata}: %{x} claims<extra></extra>",
))
fig.update_layout(title="Claim volume by domain (keyword-classified sections)",
                  xaxis_title="claims", height=430, yaxis=dict(automargin=True),
                  showlegend=False)
fig.show()

## Supporting context

### Search the corpus, and the caveats that come with the numbers

The widget below filters the full claim table by free text. Two caveats belong next to it: the domain column is derived by keyword matching on section names (sections are reused across domains, so it is a navigation aid, not a canonical assignment), and 13 claims carry no falsifier.

In [ ]:
_search = widgets.Text(
    value="",
    placeholder="filter claims by text, e.g. Thread, override, support period",
    description="Filter:",
    layout=widgets.Layout(width="640px"),
)
_out = widgets.Output()
_last_rendered = {"rows": 0}

def _render(value):
    # NOTE: Output.__exit__ swallows anything raised inside this block and
    # paints the traceback into the widget, where a notebook-level error check
    # cannot see it. The explicit try/except below re-raises so a bug here
    # fails the cell loudly instead of silently rendering nothing.
    try:
        frame = all_claims
        if value:
            needle = value.lower()
            mask = (frame["claim_text"].str.lower().str.contains(needle, na=False)
                    | frame["section"].str.lower().str.contains(needle, na=False)
                    | frame["local_id"].str.lower().str.contains(needle, na=False))
            frame = frame[mask]
        with _out:
            clear_output(wait=True)
            display(frame[["local_id", "claim_type", "confidence", "domain",
                           "workflow_stage", "section"]])
            display(HTML(f"<p>{len(frame)} of {len(all_claims)} claims</p>"))
    except Exception:
        with _out:
            clear_output(wait=True)
            display(HTML("<p><b>Filter failed:</b> see the traceback on this "
                         "cell.</p>"))
        raise
    _last_rendered["rows"] = len(frame)

_search.observe(lambda change: _render(change["new"]), names="value")
_render("")
display(widgets.VBox([_search, _out]))

# The widget body above runs inside widgets.Output, whose traceback handling
# hides failures from the kernel. This assertion is the real guard: a filter
# that never rendered anything must fail the cell.
assert _last_rendered["rows"] > 0, "claim filter rendered no rows"

## Uncertainty and gaps

### What this index does and does not certify

- **Domain counts are keyword-derived.** `viz.claim_domain` maps a claim's section name to a notebook domain. Sections such as 'Documented developments' and 'Limitations' are reused across domains, so a claim counted under 'Matter/Thread' there may belong to another line of enquiry. Counts are navigation, not taxonomy.
- **Confidence is an extraction-time judgement**, recorded per row, not a probability. A few `inference` rows sit at high confidence because the inference is well supported; that is a different claim from the inference being well measured.
- **13 claims have no falsifier** (`L030`). A claim with no stated disconfirming observation cannot be re-tested from this corpus.
- **Metric linkage is heuristic.** `viz.fetch_metrics_for_claim` links by shared section first and shared source id second, and labels which rule fired. There is no foreign key between claims and metrics in `schemas/stage2.sql`.
- **Forecast material is mixed into the corpus.** `L014` and `L031` record evidence downgrades where Stage 1 relied on vendor or analyst material; those rows sit in the `reported signal` band above.

In [ ]:
display(HTML("<h4>Claims carrying no falsifier (decision L030)</h4>"))
_missing = all_claims[all_claims["falsifier_missing"]]
show(_missing[["local_id", "claim_type", "confidence", "domain", "section"]]
     .rename(columns={"local_id": "claim", "claim_type": "type",
                      "confidence": "conf"}))
display(HTML(
    f"<p>{len(_missing)} claims carry no falsifier, matching the count in "
    f"decision <code>L030</code> "
    f"({'agrees' if len(_missing) == 13 else 'DIFFERS FROM'} the extraction log).</p>"
))

display(HTML("<h4>Metrics with conditions not stated in the source</h4>"))
_con = viz.connect_db()
try:
    _unstated = pd.DataFrame(
        _con.execute(
            "SELECT local_id, metric_name, value, unit, section FROM metric "
            "WHERE conditions_stated = FALSE ORDER BY local_id"
        ).fetchall(),
        columns=["metric", "name", "value", "unit", "section"],
    )
finally:
    _con.close()
show(_unstated)

display(HTML("<h4>Extraction decisions that downgraded or conflicted</h4>"))
show(pd.DataFrame(
    viz.connect_db().execute(
        "SELECT local_id, decision_type, description, resolution "
        "FROM extraction_decision "
        "WHERE decision_type IN ('evidence_downgrade', 'classification_conflict', "
        "                           'falsifier_absent') ORDER BY local_id"
    ).fetchall(),
    columns=["decision", "type", "description", "resolution"],
))

## Reproducibility

The claim table is read in full with `viz.fetch_all_claims`; every count on this page is a group-by over that frame, with no hand-maintained totals. Metrics with unstated conditions and the downgrade decisions come from the `unstated_conditions` view and the `extraction_decision` table respectively.

In [ ]:
# The exact queries behind every figure above, re-run verbatim.
import duckdb
con = duckdb.connect(str(ROOT / 'data' / 'smarthome.duckdb'), read_only=True)
print('database:', con.execute('SELECT current_database()').fetchone()[0])
for table in ('stage2_claim', 'metric', 'entity', 'source_mirror',
               'predicate', 'workflow_stage', 'extraction_decision'):
    total = con.execute(f'SELECT COUNT(*) FROM {table}').fetchone()[0]
    print(f'{table:22} {total:>6} rows')
con.close()